In [ ]:
PASSWORDS = {
    ",jr]&D": ["comma", "j", "r", "bracketclose", "lshift", "7", "rshift", "d"],
    "x!z29B": ["x", "rshift", "1", "z", "2", "9", "lshift", "b"],
    "D{8=aq": ["caps", "d", "bracketopen", "caps", "8", "equal", "a", "q"],
    "8%$Bbt": ["8", "caps", "5", "4", "b", "caps", "b", "t"],
    "uR.@+!": ["u", "lshift", "r", "dot", "rshift", "2", "lshift", "equal", "lshift", "1"],
    "7^qR5q@G": ["7", "lshift", "6", "q", "rshift", "r", "5", "q", "lshift", "2", "rshift", "g"],
    "a$==(L['": ["a", "lshift", "4", "equal", "equal", "lshift", "9", "rshift", "l", "bracketclose", "apos"],
    "G[)#P.6k": ["lshift", "g", "bracketopen", "caps", "0", "3", "p", "caps", "dot", "6", "k"],
    "@HDkeYd)": ["rshift", "2", "caps", "h", "d", "caps", "k", "e", "lshift", "y", "d", "rshift", "0"],
    "4vFq!G)5": ["4", "v", "lshift", "f", "q", "rshift", "1", "caps", "g", "0", "caps", "5"],
    "cMhOH{!&$S": ["c", "lshift", "m", "h", "0", "caps", "h", "bracketopen", "1", "7", "5", "s", "caps"],
    "}J7tOz$fWZ": ["lshift", "bracketclose", "rshift", "j", "7", "t", "0", "z", "lshift", "4", "f", "rshift", "w", "lshift", "z"],
    "e.};+@8RzW": ["e", "dot", "lshift", "bracketclose", "semicolon", "lshift", "equal", "rshift", "2", "8", "lshift", "r", "z", "rshift", "w"],
    "V{9Qk;JYqz": ["caps", "v", "bracketopen", "caps", "9", "lshift", "q", "k", "semicolon", "rshift", "j", "lshift", "y", "q", "z"],
    "nbPH1qZ^RD": ["n", "b", "caps", "p", "h", "caps", "1", "q", "lshift", "z", "lshift", "6", "caps", "r", "d", "caps"],
    "hU1!}{Zj3G8,": ["h", "lshift", "u", "1", "lshift", "1", "caps", "bracketclose", "bracketopen", "z", "caps", "j", "3", "rshift", "g", "8", "comma"],
    "Dc&mH&bNr1Rb": ["lshift", "d", "c", "lshift", "7", "m", "caps", "h", "7", "caps", "b", "lshift", "n", "r", "1", "rshift", "r", "b"],
    ".by9ClNLlS(V": ["dot", "b", "y", "9", "lshift", "c", "l", "rshift", "n", "lshift", "l", "l", "caps", "s", "9", "v", "caps"],
    "QIhH8B5JmRng": ["caps", "q", "i", "caps", "h", "lshift", "h", "8", "rshift", "b", "5", "lshift", "j", "m", "lshift", "r", "n", "g"],
    "!Wy%'}Ghe'+9": ["caps", "1", "w", "caps", "y", "lshift", "5", "apos", "rshift", "bracketclose", "rshift", "g", "h", "e", "apos", "lshift", "equal", "9"]
}

In [ ]:
import torch
import torch.nn.functional as F
import random
from tqdm import tqdm_notebook
import torchaudio
import numpy as np
import pandas as pd
import json

import os
import sys

sys.path.append(os.path.abspath("./../src"))

from utils.loading import AudioDataset, normalize_waveform
from models.coatnet import MyCoAtNet

from dotenv import load_dotenv
load_dotenv()

np.random.seed(42)

ROOT = f"{os.getenv("ARTICLE_DATA_ROOT")}test"
MODEL_ROOT = os.getenv("ARTICLE_MODEL_ROOT")
MODEL_NAME = 'coatnet_bigger_adam_CosineAnnealingLR_lr_0.0001_partial_wd_0.05_special_keys_True_all_128_in_lr_True_unmapped'


In [ ]:
def predict_file_log_probs(model, test_dataset, file_path):
    """
    Returns the LogSoftmax distribution for a single audio file.
    Shape: [1, num_classes]
    """
    waveform, sr = torchaudio.load(file_path)

    if getattr(test_dataset, "noise_reduction", False):
        waveform = test_dataset.apply_noise_reduction(waveform, sr)

    waveform = normalize_waveform(waveform, test_dataset.dataset, root=os.getenv("ARTICLE_ROOT")).to(torch.float32)
    
    if waveform.shape[1] / sr > 0.5:
        spectrogram = test_dataset.transform_long(waveform)
    else:
        spectrogram = test_dataset.transform_short(waveform)

    spectrogram = spectrogram.unsqueeze(0)
    
    with torch.no_grad():
        logits = model(spectrogram)
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        log_probs = F.log_softmax(logits, dim=1) 
        
    return log_probs

In [ ]:
def beam_search_decoder(log_probs_list, width=5, idx_to_class=None):
    """
    Finds the N (width) most likely sequences using Beam Search.
    
    Args:
        log_probs_list: List of tensors, where each tensor is [1, num_classes] for one keystroke.
        width: Number of top sequences to return (N).
        idx_to_class: Dict to map indices back to key strings.
        
    Returns:
        List of dicts: [{'sequence': ['a', 'b'...], 'score': -1.23}, ...]
    """
    beam = [(0.0, [])]
    
    for step_log_probs in log_probs_list:
        candidates = []
        
        step_log_probs = step_log_probs.squeeze(0)
        top_k_step_probs, top_k_step_indices = torch.topk(step_log_probs, k=min(20, len(step_log_probs)))
        
        for total_score, current_seq in beam:
            for prob, idx in zip(top_k_step_probs, top_k_step_indices):
                new_score = total_score + prob.item()
                new_seq = current_seq + [idx.item()]
                candidates.append((new_score, new_seq))
        
        ordered = sorted(candidates, key=lambda x: x[0], reverse=True)
        beam = ordered[:width]
    
    results = []
    for score, seq_indices in beam:
        mapped_seq = [idx_to_class[idx] for idx in seq_indices]
                
        results.append({
            "sequence": mapped_seq,
            "score": score
        })
        
    return results

In [ ]:
def process_sentences_with_rank(data_root, model_root, model_name, search_width=1000):
    """
    Returns the rank of the correct password among the top N (search_width) predictions.
    """
    keys = model_name.split("_")
    model_type = keys[0]
    checkpoint_path = f"{model_root}/{model_name}/model_checkpoints/"
    config_path = f"{model_root}/{model_name}/config.json"

    with open(config_path, "r") as f:
        config = json.load(f)

    model_name_in_config = config["model"]
    model_params = config["model_params"]
    
    if model_type == "coatnet":
        config["image_size"] = 64
        config["model_configs"][model_type][model_params]["image_size"] = 64
    elif model_type == "moat":
        config["img_size"] = 64
        config["model_configs"][model_type][model_params]["img_size"] = 64

    special_keys = config.get("special_keys", False)
    checkpoint_file = os.path.join(checkpoint_path, os.listdir(checkpoint_path)[0])
    checkpoint = torch.load(checkpoint_file, map_location="cpu")
    num_classes = len(config["class_encoding"])

    if model_type == "coatnet":
        model = MyCoAtNet(num_classes=num_classes, **config["model_configs"][model_name_in_config][model_params])
    else:
        model = MOAT(num_classes=num_classes, **config["model_configs"][model_name_in_config][model_params])

    model.load_state_dict(checkpoint)
    model.eval()

    test_dataset = AudioDataset(
        root=data_root,
        dataset=config["dataset"],
        transform_aug=False,
        special_keys=special_keys,
        image_size=config["image_size"],
        exclude_few_special_keys=True,
        sample_rate=22000,
        class_idx=config["class_encoding"],
    )

    idx_to_class = {idx: cls for cls, idx in test_dataset.class_to_idx.items()}
    class_to_idx = test_dataset.class_to_idx # Needed to find target indices

    files_by_label = {}
    for path, label in test_dataset.file_paths:
        files_by_label.setdefault(label, []).append(path)

    results = {}
    
    for i in tqdm_notebook(range(5), desc="Total Progress (Seeds)", unit="seed"):
        random.seed(i)
        
        for target_sentence_str, target_keys in PASSWORDS.items():
            
            selected_paths = []
            valid_target = True
            target_indices = []

            for key in target_keys:
                candidates = files_by_label.get(key, [])
                if candidates:
                    selected_paths.append(random.choice(candidates))
                else:
                    print(f"Warning: No audio file found for key '{key}'")
                    valid_target = False
                
                if key in class_to_idx:
                    target_indices.append(class_to_idx[key])
                else:
                    print(f"Warning: Key '{key}' not in model class_encoding")
                    valid_target = False

            if not valid_target:
                continue

            sequence_log_probs = []
            for file_path in selected_paths:
                log_probs = predict_file_log_probs(model, test_dataset, file_path)
                sequence_log_probs.append(log_probs)

            top_sequences = beam_search_decoder(
                sequence_log_probs, 
                width=search_width, 
                idx_to_class=idx_to_class
            )

            rank = "> " + str(search_width) 
            
            best_score = top_sequences[0]['score'] if top_sequences else -9999

            for idx, item in enumerate(top_sequences):
                if item['sequence'] == target_keys:
                    rank = idx + 1
                    break

            unique_key = f"{target_sentence_str}_{i}"

            results[unique_key] = {
                "Password": target_sentence_str,
                "Length": len(target_sentence_str),
                "Run": i,
                "Rank": rank, 
                "Best Score": best_score,
                "Actual Keys": target_keys,
                "Best Prediction": top_sequences[0]['sequence'] if top_sequences else [],
            }

    return results


In [ ]:
results = process_sentences_with_rank(ROOT, MODEL_ROOT, MODEL_NAME, search_width=1000)
df = pd.DataFrame(list(results.values()))

In [18]:
df = df.sort_values(by=["Length", "Password", "Run"])
df.to_csv("../results/password_recovery.csv")

In [ ]:
stats_df = df.groupby(["Length", "Password"])[['Rank']].agg(['min', 'max', 'mean', 'median'])

stats_df.columns = ['_'.join(col).strip() for col in stats_df.columns.values]
stats_df = stats_df.reset_index()

stats_df.to_csv("../results/password_recovery_stats.csv", index=False)

In [ ]:
stats_df = df.groupby(["Length"])[['Rank']].agg(['min', 'max', 'mean', 'median'])

stats_df.columns = ['_'.join(col).strip() for col in stats_df.columns.values]
stats_df = stats_df.reset_index()

stats_df.to_csv("../results/password_recovery_stats_2.csv", index=False)